In [1]:
import subprocess
import sys

def install_if_missing(package, import_name=None):
    """Install package if not already available."""
    try:
        __import__(import_name or package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

# Core dependencies
install_if_missing("transformers")
install_if_missing("peft")
install_if_missing("accelerate")
install_if_missing("datasets")
install_if_missing("trl")
install_if_missing("tqdm")
install_if_missing("wandb")

print("\n" + "=" * 70)
print("✅ All packages ready!")
print("=" * 70)

# Verify GPU
import torch
print(f"\n GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")

✅ transformers already installed
✅ peft already installed
✅ accelerate already installed
✅ datasets already installed
📦 Installing trl...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.9 MB/s eta 0:00:00
✅ trl installed successfully
✅ tqdm already installed
✅ wandb already installed

✅ All packages ready!

 GPU Available: True
   GPU Name: Tesla T4
   VRAM Total: 15.6 GB
   CUDA Version: 12.8


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [4]:
# =============================================================================
# STAGE 1: Confidence-Gate Calibration (standalone — no training)
# Loads student + teacher, samples T=1.0 rollouts, measures teacher entropy,
# and prints the calibration values you need before launching training.
# =============================================================================

import os
import re
import json
import math
import random
import signal
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login as hf_login, list_repo_files, hf_hub_download

# ========================= CONFIG =========================
STUDENT_ID = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# Calibration settings — matched to what training will use
CALIB_TEMPERATURE = 1.2     # IMPORTANT: not 1.2, this is the pivot
CALIB_TOP_P = 0.99
CALIB_N_PROMPTS = 30
CALIB_SAMPLES_PER_PROMPT = 2
CALIB_MAX_NEW_TOKENS = 512

# Same model-loading config as training
USE_MATH = False                 # match whatever v8_pool you'll train on
MATH_LIMIT = 5000
GSM8K_LIMIT = 6000

# ========================= HF TOKEN (Kaggle secrets) =========================
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    HF_TOKEN = _secrets.get_secret("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing. Add it as a Kaggle secret named 'HF_TOKEN'.")

hf_login(token=HF_TOKEN)
print(f"✅ HF login OK")

# ========================= TOKENIZER =========================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ========================= DATA POOL =========================
print("Loading data pool...")

def parse_number(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip()
    m = re.findall(r"-?\d+\.?\d*/\d+\.?\d*|-?\d+\.?\d*", s)
    if not m: return None
    tok = m[-1]
    try:
        if "/" in tok:
            num, den = tok.split("/")
            return float(num) / float(den)
        return float(tok)
    except:
        return None

def extract_boxed_content(text):
    idx = text.rfind("\\boxed")
    if idx == -1: return None
    i = text.find("{", idx)
    if i == -1: return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{": depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0: return text[i+1:j]
    return None

def load_gsm8k(limit=6000):
    print("  Loading GSM8K...")
    ds = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    data = []
    for ex in ds:
        gt = ex["answer"].split("####")[-1].replace(",", "").strip()
        if parse_number(gt) is None:
            continue
        data.append({"question": ex["question"], "answer": gt, "source": "GSM8K"})
    print(f"    Loaded {len(data)} GSM8K problems.")
    return data

def load_math(limit=5000):
    print("  Loading MATH (Levels 3-5)...")
    try:
        ds = load_dataset("qwedsacf/competition_math", split="train", token=HF_TOKEN)
    except Exception:
        ds = load_dataset("hendrycks/competition_math", split="train", token=HF_TOKEN)

    def level_filter(ex):
        lv = ex['level']
        if isinstance(lv, str):
            m = re.search(r'(\d+)', lv)
            return int(m.group(1)) >= 3 if m else False
        return int(lv) >= 3

    ds = ds.filter(level_filter)
    ds = ds.filter(lambda ex: len(ex["solution"]) < 800)
    ds = ds.filter(lambda ex: len(ex["problem"]) < 800)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))

    data = []
    for ex in ds:
        gt = extract_boxed_content(ex["solution"])
        if gt is None:
            continue
        data.append({"question": ex["problem"], "answer": gt, "source": "MATH"})
    print(f"    Loaded {len(data)} MATH problems.")
    return data

v8_pool = []
if USE_MATH:
    v8_pool.extend(load_math(MATH_LIMIT))
v8_pool.extend(load_gsm8k(GSM8K_LIMIT))
print(f"Total pool size: {len(v8_pool)} problems")

# ========================= PROMPT =========================
def build_prompt(question):
    user_content = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}"
    )
    chat = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

# ========================= MEASURE MAX_TOTAL_LEN =========================
print("\nMeasuring prompt length distribution...")
_prompt_lens = [
    len(tokenizer(build_prompt(ex["question"]), add_special_tokens=False).input_ids)
    for ex in v8_pool[:2000]   # subsample for speed
]
_prompt_lens.sort()
_p99 = _prompt_lens[int(0.99 * (len(_prompt_lens) - 1))]
MAX_TOTAL_LEN = _p99 + CALIB_MAX_NEW_TOKENS
print(f"  prompt len: min={_prompt_lens[0]}, "
      f"median={_prompt_lens[len(_prompt_lens)//2]}, "
      f"p99={_p99}, max={_prompt_lens[-1]}")
print(f"  MAX_TOTAL_LEN set to {MAX_TOTAL_LEN}")

# ========================= MODELS =========================
print("\nLoading models...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

# Student — GPU 0. We only need base student for calibration (no LoRA needed,
# since we sample rollouts from the *starting* policy).
student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_config,
    device_map={"": "cuda:0"}, torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
student.eval()

# Teacher — GPU 1
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, quantization_config=bnb_config,
    device_map={"": "cuda:1"}, torch_dtype=torch.float16,
    trust_remote_code=True,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f"GPU0: {torch.cuda.memory_allocated('cuda:0')/1e9:.2f} GB")
print(f"GPU1: {torch.cuda.memory_allocated('cuda:1')/1e9:.2f} GB")

# ========================= get_token_logps (with per-token entropy) =========================
CHUNK_SIZE = 1

def get_token_logps(model, input_ids, attention_mask, prompt_lens,
                    return_entropy=False, return_token_entropy=False, device=None):
    if device is not None:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)
    else:
        device = next(model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)

    all_token_logps, all_masks = [], []
    entropy_sum, entropy_count = 0.0, 0
    all_token_entropy = [] if return_token_entropy else None

    for i in range(0, input_ids.shape[0], CHUNK_SIZE):
        chunk_ids = input_ids[i:i+CHUNK_SIZE]
        chunk_mask = attention_mask[i:i+CHUNK_SIZE]
        chunk_prompts = prompt_lens[i:i+CHUNK_SIZE]

        out = model(input_ids=chunk_ids, attention_mask=chunk_mask)
        logits = out.logits[:, :-1, :]
        targets = chunk_ids[:, 1:]

        target_logits = torch.gather(logits, 2, targets.unsqueeze(-1)).squeeze(-1)
        logsumexp = torch.logsumexp(logits, dim=-1)
        token_logps = target_logits - logsumexp

        seq_len = chunk_ids.shape[1] - 1
        pos = torch.arange(seq_len, device=device).unsqueeze(0)
        comp_mask = (pos >= (chunk_prompts.unsqueeze(1) - 1)).long() * chunk_mask[:, 1:]

        if return_entropy or return_token_entropy:
            probs = torch.softmax(logits.float(), dim=-1)
            token_entropy = -(probs * torch.log(probs.clamp_min(1e-10))).sum(dim=-1)
            if return_entropy:
                entropy_sum += (token_entropy * comp_mask).sum().item()
                entropy_count += comp_mask.sum().item()
            if return_token_entropy:
                all_token_entropy.append(token_entropy)

        all_token_logps.append(token_logps)
        all_masks.append(comp_mask)
        del out, logits

    token_logps = torch.cat(all_token_logps, dim=0).float()
    comp_mask = torch.cat(all_masks, dim=0)

    result = [token_logps, comp_mask]
    if return_entropy:
        result.append(entropy_sum / max(entropy_count, 1))
    if return_token_entropy:
        result.append(torch.cat(all_token_entropy, dim=0))
    return tuple(result)

# ========================= CALIBRATION LOOP =========================
print("\n" + "=" * 80)
print(f"STAGE 1: Confidence-gate calibration @ T={CALIB_TEMPERATURE}")
print("=" * 80)
print(f"Sampling {CALIB_N_PROMPTS} problems × {CALIB_SAMPLES_PER_PROMPT} rollouts each")

calib_entropies = []
calib_logps      = []   # also useful to look at rho / log_rho stats

student.eval()
with torch.no_grad():
    sampled = random.sample(range(len(v8_pool)), min(CALIB_N_PROMPTS, len(v8_pool)))
    for idx, pool_idx in enumerate(sampled):
        ex = v8_pool[pool_idx]
        prompt_text = build_prompt(ex["question"])
        prompt_ids = tokenizer(prompt_text, return_tensors="pt",
                                add_special_tokens=False).input_ids.to("cuda:0")
        plen = prompt_ids.shape[1]

        # Generate N samples from the student at T=1.0
        gen_inputs = prompt_ids.repeat(CALIB_SAMPLES_PER_PROMPT, 1)
        gen_attn = torch.ones_like(gen_inputs)
        generated = student.generate(
            gen_inputs, attention_mask=gen_attn,
            max_new_tokens=CALIB_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=CALIB_TEMPERATURE, top_p=CALIB_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )

        for row in generated:
            row_t = row[:MAX_TOTAL_LEN].unsqueeze(0)
            attn = torch.ones_like(row_t)
            plen_t = torch.tensor([plen], device="cuda:0")

            # Teacher forward on cuda:1, request per-token entropy
            t_logps, comp_mask, tok_ent = get_token_logps(
                teacher, row_t, attn, plen_t,
                device="cuda:1",
                return_token_entropy=True,
            )
            valid = comp_mask.bool()[0]
            if valid.sum() > 0:
                calib_entropies.append(tok_ent[0][valid].float().cpu())
                calib_logps.append(t_logps[0][valid].float().cpu())

        del generated, gen_inputs
        torch.cuda.empty_cache()

        if (idx + 1) % 5 == 0:
            print(f"  processed {idx+1}/{len(sampled)} problems")

# ========================= AGGREGATE & PRINT =========================
print("\n" + "=" * 80)
print("CALIBRATION RESULTS")
print("=" * 80)

all_H = torch.cat(calib_entropies).float()
all_LP = torch.cat(calib_logps).float()


N_TOKENS = len(all_H)
print(f"\nTotal completion tokens measured: {N_TOKENS:,}")

# --- Basic stats on teacher entropy ---
H_mean   = all_H.mean().item()
H_median = all_H.median().item()
H_std    = all_H.std().item()
H_p10    = torch.quantile(all_H, 0.10).item()
H_p25    = torch.quantile(all_H, 0.25).item()
H_p75    = torch.quantile(all_H, 0.75).item()
H_p90    = torch.quantile(all_H, 0.90).item()
H_min    = all_H.min().item()
H_max    = all_H.max().item()

print(f"\n--- Teacher per-token entropy H (nats) ---")
print(f"  min    = {H_min:.4f}")
print(f"  p10    = {H_p10:.4f}")
print(f"  p25    = {H_p25:.4f}")
print(f"  median = {H_median:.4f}   <-- ENTROPY_THRESHOLD")
print(f"  mean   = {H_mean:.4f}")
print(f"  p75    = {H_p75:.4f}")
print(f"  p90    = {H_p90:.4f}")
print(f"  max    = {H_max:.4f}")
print(f"  std    = {H_std:.4f}")

# --- Calibrated parameters ---
ENTROPY_THRESHOLD = H_p75
GATE_TEMP = H_std / 4.0

print(f"\n--- Calibrated gate parameters ---")
print(f"  ENTROPY_THRESHOLD = {ENTROPY_THRESHOLD:.4f}   (= median H)")
print(f"  GATE_TEMP         = {GATE_TEMP:.4f}          (= std H / 2)")

# --- Evaluate the gate function on this data ---
# confidence_gate = sigmoid((threshold - H) / gate_temp)
c_values = torch.sigmoid((ENTROPY_THRESHOLD - all_H) / GATE_TEMP)

c_mean   = c_values.mean().item()
c_median = c_values.median().item()
c_std    = c_values.std().item()
c_p10    = torch.quantile(c_values, 0.10).item()
c_p90    = torch.quantile(c_values, 0.90).item()

frac_c_gt_09 = (c_values > 0.9).float().mean().item()
frac_c_lt_05 = (c_values < 0.5).float().mean().item()
frac_c_mid   = ((c_values >= 0.5) & (c_values <= 0.9)).float().mean().item()
frac_c_lt_01 = (c_values < 0.1).float().mean().item()
frac_c_gt_099 = (c_values > 0.99).float().mean().item()

print(f"\n--- Confidence gate c = sigmoid((H_thr - H)/T_gate) on measured H ---")
print(f"  mean   c = {c_mean:.4f}")
print(f"  median c = {c_median:.4f}")
print(f"  std    c = {c_std:.4f}")
print(f"  p10    c = {c_p10:.4f}")
print(f"  p90    c = {c_p90:.4f}")
print(f"\n  fraction c > 0.99  = {frac_c_gt_099:.4f}")
print(f"  fraction c > 0.90  = {frac_c_gt_09:.4f}")
print(f"  fraction 0.5..0.9  = {frac_c_mid:.4f}")
print(f"  fraction c < 0.50  = {frac_c_lt_05:.4f}")
print(f"  fraction c < 0.10  = {frac_c_lt_01:.4f}")

# --- DeepSeek-style band check ---
print(f"\n--- DeepSeek band check ---")
print(f"  mean c = {c_mean:.3f}")
if c_mean > 0.9:
    band = "NO-OP (teacher already very confident; gate barely acts)"
elif c_mean >= 0.5:
    band = "MEANINGFUL (good training regime)"
else:
    band = "AGGRESSIVE / RISKY (teacher often uncertain)"

print(f"  >> Band: {band}")

# --- Sanity: how much does the gate actually attenuate rho_tilde? ---
# If rho_tilde = 2 (teacher strongly prefers this token), then
#   w = 1 + c*(rho_tilde - 1) = 1 + c
# So c=1 -> w=2 (full V8), c=0.5 -> w=1.5, c=0 -> w=1 (pure RL)
print(f"\n--- Effect on weights (illustration, rho_tilde=2) ---")
print(f"  c=1.0  -> w = 2.00  (full teacher influence)")
print(f"  c=0.5  -> w = 1.50")
print(f"  c=0.0  -> w = 1.00  (pure RL)")
print(f"  Observed mean c implies mean attenuation: "
      f"w = {1.0 + c_mean:.2f} when rho_tilde=2")

# --- Also check the raw teacher logps for context ---
lp_mean = all_LP.mean().item()
lp_std  = all_LP.std().item()
print(f"\n--- For reference: teacher log-prob of sampled tokens ---")
print(f"  mean log p = {lp_mean:.4f}  (i.e. ~{math.exp(lp_mean):.4f} geometric mean prob)")
print(f"  std log p  = {lp_std:.4f}")

# ========================= SAVE =========================
CALIB_PATH = "gate_calibration.json"
calib_data = {
    "ENTROPY_THRESHOLD": ENTROPY_THRESHOLD,
    "GATE_TEMP": GATE_TEMP,
    "entropy_stats": {
        "n_tokens": N_TOKENS,
        "min": H_min, "p10": H_p10, "p25": H_p25,
        "median": H_median, "mean": H_mean,
        "p75": H_p75, "p90": H_p90, "max": H_max,
        "std": H_std,
    },
    "gate_stats": {
        "mean_c": c_mean, "median_c": c_median, "std_c": c_std,
        "p10_c": c_p10, "p90_c": c_p90,
        "frac_c_gt_099": frac_c_gt_099,
        "frac_c_gt_09": frac_c_gt_09,
        "frac_c_mid": frac_c_mid,
        "frac_c_lt_05": frac_c_lt_05,
        "frac_c_lt_01": frac_c_lt_01,
    },
    "deepseek_band": band,
    "calibration_temperature": CALIB_TEMPERATURE,
    "calibration_n_prompts": CALIB_N_PROMPTS,
    "calibration_samples_per_prompt": CALIB_SAMPLES_PER_PROMPT,
    "max_total_len_used": MAX_TOTAL_LEN,
}
with open(CALIB_PATH, "w") as f:
    json.dump(calib_data, f, indent=2)

print(f"\n✅ Saved calibration to {CALIB_PATH}")

# ========================= RECOMMENDATION =========================
print("\n" + "=" * 80)
print("NEXT STEP")
print("=" * 80)
if 0.5 <= c_mean <= 0.9:
    print(f"✅ mean c = {c_mean:.3f} is in the MEANINGFUL band.")
    print("   -> Proceed to launch the three training arms at T=1.0.")
    print("   -> In training, load these values:")
    print(f"        ENTROPY_THRESHOLD = {ENTROPY_THRESHOLD:.4f}")
    print(f"        GATE_TEMP         = {GATE_TEMP:.4f}")
elif c_mean > 0.9:
    print(f"⚠️  mean c = {c_mean:.3f} is in the NO-OP band.")
    print("   -> The gate barely modulates anything at these settings.")
    print("   -> Options:")
    print("      a) Lower ENTROPY_THRESHOLD (e.g. subtract 0.2 from median) to")
    print("         push more tokens into the uncertain regime.")
    print("      b) Accept that at T=1.0 the teacher is very confident and")
    print("         consider running only DRL vs DRL-gated (skip the 'no-op'")
    print("         hypothesis test as trivially satisfied).")
else:
    print(f"⚠️  mean c = {c_mean:.3f} is in the AGGRESSIVE/RISKY band.")
    print("   -> The gate acts on almost every token (teacher is uncertain).")
    print("   -> Options:")
    print("      a) Raise GATE_TEMP (e.g. ×2) to soften the gate.")
    print("      b) Raise ENTROPY_THRESHOLD so more tokens fall below it.")
    print("      c) Keep as-is if you *want* an aggressive regime — but")
    print("         note that then 'DRL' and 'DRL-gated' will diverge a lot.")
print("=" * 80)

✅ HF login OK
Loading tokenizer...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

GPU0: 1.03 GB
GPU1: 5.71 GB

STAGE 1: Confidence-gate calibration @ T=1.2
Sampling 30 problems × 2 rollouts each
  processed 5/30 problems
  processed 10/30 problems
  processed 15/30 problems
  processed 20/30 problems
  processed 25/30 problems
  processed 30/30 problems

CALIBRATION RESULTS

Total completion tokens measured: 15,448

--- Teacher per-token entropy H (nats) ---
  min    = 0.0000
  p10    = 0.0036
  p25    = 0.0539
  median = 0.6494   <-- ENTROPY_THRESHOLD
  mean   = 1.1226
  p75    = 1.5737
  p90    = 3.1244
  max    = 7.0450
  std    = 1.3938

--- Calibrated gate parameters ---
  ENTROPY_THRESHOLD = 1.5737   (= median H)
  GATE_TEMP         = 0.3484          (= std H / 2)

--- Confidence gate c = sigmoid((H_thr - H)/T_gate) on measured H ---
  mean   c = 0.7188
  median c = 0.9342
  std    c = 0.3666
  p10    c = 0.0115
  p90    c = 0.9891

  fraction c > 0.99  = 0.0000
  fraction c > 0.90  = 0.5616
  fraction 0.5..0.9  = 0.1884
  fraction c < 0.50  = 0.2500
  fractio

In [6]:
# =============================================================================
# STAGE 1.5: Pre-training diagnostic — is the gate aligned with the signal?
# =============================================================================
# Question: on positive-advantage tokens, is the confidence gate c
# correlated with the magnitude of the teacher signal (rho_tilde - 1)?
#
# If yes  -> gate suppresses noisy tokens, amplifies signal-rich tokens: good.
# If no   -> gate is orthogonal, still okay but not amplifying.
# If neg  -> gate suppresses the teacher where it actually has something to
#            say: bad, would need formula change before training.
# =============================================================================

import os
import re
import json
import math
import random
import signal
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login as hf_login

# ========================= CONFIG =========================
STUDENT_ID = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# OPTIONAL: start diagnostic from V12's trained policy instead of base student.
# Set to the HF repo / local path of your V12 adapter, or None for base.
V12_ADAPTER_PATH = None   # e.g. "./hf_checkpoints/v12/step80"

DIAG_TEMPERATURE = 1.2
DIAG_TOP_P = 0.99
DIAG_N_PROBLEMS = 20          # small sample
DIAG_GROUP_SIZE = 10          # rollouts per problem
DIAG_MAX_NEW_TOKENS = 512
DIAG_MAX_TOTAL_LEN = 1200     # generous; you can also reuse your measured value

# Use MATH in the pool for a more interesting difficulty mix
USE_MATH = False              # match whatever you'll actually train on
MATH_LIMIT = 5000
GSM8K_LIMIT = 6000

CALIB_PATH = "gate_calibration.json"

# ========================= AUTH =========================
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing.")
hf_login(token=HF_TOKEN)
print("✅ HF login OK")

# ========================= LOAD CALIBRATION =========================
with open(CALIB_PATH) as f:
    calib = json.load(f)
ENTROPY_THRESHOLD = calib["ENTROPY_THRESHOLD"]
GATE_TEMP         = calib["GATE_TEMP"]
print(f"Loaded calibration: H* = {ENTROPY_THRESHOLD:.4f}, tau = {GATE_TEMP:.4f}, "
      f"calib T = {calib.get('calibration_temperature')}")

# ========================= HELPERS =========================
def parse_number(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip()
    m = re.findall(r"-?\d+\.?\d*/\d+\.?\d*|-?\d+\.?\d*", s)
    if not m: return None
    tok = m[-1]
    try:
        if "/" in tok:
            num, den = tok.split("/")
            return float(num) / float(den)
        return float(tok)
    except:
        return None

def extract_boxed_content(text):
    idx = text.rfind("\\boxed")
    if idx == -1: return None
    i = text.find("{", idx)
    if i == -1: return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{": depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0: return text[i+1:j]
    return None

def extract_answer_regex(text):
    if not text: return None
    boxed = extract_boxed_content(text)
    if boxed is not None:
        v = parse_number(boxed)
        if v is not None: return v
    for pat in [
        r"(?:final\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:the\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
        r"(?:therefore|thus|so)\s*,?\s*(?:the\s+)?answer\s*(?:is|=|:)\s*([^\n.]+)",
    ]:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            v = parse_number(matches[-1])
            if v is not None: return v
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if sentences:
        v = parse_number(sentences[-1])
        if v is not None: return v
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if nums:
        v = parse_number(nums[-1])
        if v is not None: return v
    return None

def compute_reward_simple(text, gt):
    """Lightweight diagnostic reward: boxed + regex, no math_verify calls."""
    pred = extract_answer_regex(text)
    gt_num = parse_number(gt)
    if pred is None or gt_num is None:
        return 0.0
    return 1.0 if abs(pred - gt_num) < 1e-4 else 0.0

# ========================= LOAD DATA =========================
print("Loading data...")
def load_gsm8k(limit):
    ds = load_dataset("openai/gsm8k", "main", split="train", token=HF_TOKEN)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    out = []
    for ex in ds:
        gt = ex["answer"].split("####")[-1].replace(",", "").strip()
        if parse_number(gt) is None: continue
        out.append({"question": ex["question"], "answer": gt, "source": "GSM8K"})
    return out

def load_math(limit):
    try:
        ds = load_dataset("qwedsacf/competition_math", split="train", token=HF_TOKEN)
    except Exception:
        ds = load_dataset("hendrycks/competition_math", split="train", token=HF_TOKEN)
    def lf(ex):
        lv = ex['level']
        if isinstance(lv, str):
            m = re.search(r'(\d+)', lv)
            return int(m.group(1)) >= 3 if m else False
        return int(lv) >= 3
    ds = ds.filter(lf)
    ds = ds.filter(lambda ex: len(ex["solution"]) < 800 and len(ex["problem"]) < 800)
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    out = []
    for ex in ds:
        gt = extract_boxed_content(ex["solution"])
        if gt is None: continue
        out.append({"question": ex["problem"], "answer": gt, "source": "MATH"})
    return out

pool = []
if USE_MATH:
    pool.extend(load_math(MATH_LIMIT))
pool.extend(load_gsm8k(GSM8K_LIMIT))
print(f"Pool: {len(pool)} problems")

# ========================= TOKENIZER + MODELS =========================
print("Loading tokenizer + models...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb,
    device_map={"": "cuda:0"}, torch_dtype=torch.bfloat16, trust_remote_code=True,
)
if V12_ADAPTER_PATH is not None:
    print(f"Loading V12 adapter from {V12_ADAPTER_PATH}")
    student = PeftModel.from_pretrained(student, V12_ADAPTER_PATH)
student.eval()

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, quantization_config=bnb,
    device_map={"": "cuda:1"}, torch_dtype=torch.float16, trust_remote_code=True,
)
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False

print(f"GPU0: {torch.cuda.memory_allocated('cuda:0')/1e9:.2f} GB, "
      f"GPU1: {torch.cuda.memory_allocated('cuda:1')/1e9:.2f} GB")

# ========================= LOGPS (per-token entropy) =========================
CHUNK_SIZE = 1

def get_token_logps(model, input_ids, attention_mask, prompt_lens,
                    return_token_entropy=False, device=None):
    if device is not None:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)
    else:
        device = next(model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        prompt_lens = prompt_lens.to(device)

    all_logps, all_masks, all_ent = [], [], ([] if return_token_entropy else None)

    for i in range(0, input_ids.shape[0], CHUNK_SIZE):
        cids = input_ids[i:i+CHUNK_SIZE]
        cmask = attention_mask[i:i+CHUNK_SIZE]
        cplen = prompt_lens[i:i+CHUNK_SIZE]

        out = model(input_ids=cids, attention_mask=cmask)
        logits = out.logits[:, :-1, :]
        targets = cids[:, 1:]

        tgt = torch.gather(logits, 2, targets.unsqueeze(-1)).squeeze(-1)
        lse = torch.logsumexp(logits, dim=-1)
        token_logps = tgt - lse

        seq_len = cids.shape[1] - 1
        pos = torch.arange(seq_len, device=device).unsqueeze(0)
        comp_mask = (pos >= (cplen.unsqueeze(1) - 1)).long() * cmask[:, 1:]

        if return_token_entropy:
            probs = torch.softmax(logits.float(), dim=-1)
            tok_ent = -(probs * torch.log(probs.clamp_min(1e-10))).sum(dim=-1)
            all_ent.append(tok_ent)

        all_logps.append(token_logps)
        all_masks.append(comp_mask)
        del out, logits

    token_logps = torch.cat(all_logps, dim=0).float()
    comp_mask   = torch.cat(all_masks, dim=0)
    result = [token_logps, comp_mask]
    if return_token_entropy:
        result.append(torch.cat(all_ent, dim=0))
    return tuple(result)

def build_prompt(question):
    user_content = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{question}"
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        add_generation_prompt=True, tokenize=False,
    )

def group_normalize(rewards, group_ids):
    adv = torch.zeros_like(rewards)
    for g in group_ids.unique():
        mask = group_ids == g
        r = rewards[mask]
        std = r.std()
        if std > 0:
            adv[mask] = (r - r.mean()) / (std + 1e-4)
    return adv

# ========================= DIAGNOSTIC LOOP =========================
print("\n" + "=" * 80)
print(f"DIAGNOSTIC: rho_tilde vs confidence gate @ T={DIAG_TEMPERATURE}")
print("=" * 80)

# Per-token records for A>0 tokens
rho_pos_list   = []   # rho_tilde values on positive-adv tokens
c_pos_list     = []   # confidence gate c on those tokens
w_pos_list     = []   # resulting w = 1 + c*(rho_tilde - 1)
adv_pos_list   = []   # advantage values (to sanity-check sign)
src_pos_list   = []   # source name

student.eval()
with torch.no_grad():
    sampled_idx = random.sample(range(len(pool)), DIAG_N_PROBLEMS)

    for pi, idx in enumerate(sampled_idx):
        ex = pool[idx]
        prompt_text = build_prompt(ex["question"])
        prompt_ids = tokenizer(prompt_text, return_tensors="pt",
                               add_special_tokens=False).input_ids.to("cuda:0")
        plen = prompt_ids.shape[1]

        # Generate G rollouts
        gen_in = prompt_ids.repeat(DIAG_GROUP_SIZE, 1)
        gen_att = torch.ones_like(gen_in)
        gen = student.generate(
            gen_in, attention_mask=gen_att,
            max_new_tokens=DIAG_MAX_NEW_TOKENS,
            do_sample=True, temperature=DIAG_TEMPERATURE, top_p=DIAG_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )

        # Score
        gt = ex["answer"]
        rewards = []
        rows = []
        for row in gen:
            row_t = row[:DIAG_MAX_TOTAL_LEN]
            rows.append(row_t)
            text = tokenizer.decode(row_t[plen:], skip_special_tokens=True)
            rewards.append(compute_reward_simple(text, gt))

        if len(set(rewards)) == 1:
            # all-correct or all-wrong -> no advantage variance, skip
            del gen, gen_in
            torch.cuda.empty_cache()
            continue

        rewards_t = torch.tensor(rewards, dtype=torch.float32, device="cuda:0")
        group_ids = torch.zeros(len(rows), dtype=torch.long, device="cuda:0")
        adv = group_normalize(rewards_t, group_ids)   # single-group normalize
        adv_cpu = adv.cpu()

        # Build padded batch (one group, all rows at once — small)
        max_len = max(r.shape[0] for r in rows)
        ids = torch.full((len(rows), max_len), tokenizer.pad_token_id,
                         dtype=torch.long, device="cuda:0")
        am  = torch.zeros((len(rows), max_len), dtype=torch.long, device="cuda:0")
        for i, r in enumerate(rows):
            ids[i, :r.shape[0]] = r
            am[i, :r.shape[0]] = 1
        plen_t = torch.tensor([plen] * len(rows), device="cuda:0")

        # Student forward (old policy) on cuda:0
        old_logps, comp_mask = get_token_logps(student, ids, am, plen_t)

        # Teacher forward on cuda:1 with per-token entropy
        teach_logps, _, teach_ent = get_token_logps(
            teacher, ids, am, plen_t,
            device="cuda:1", return_token_entropy=True,
        )
        teach_logps = teach_logps.cpu()
        teach_ent   = teach_ent.cpu()
        old_logps   = old_logps.cpu()
        comp_mask   = comp_mask.cpu()

        # Compute rho_tilde (same as training)
        log_rho = teach_logps - old_logps
        rho = torch.exp(log_rho)
        rho_bar = torch.clamp(rho, 1.0/3.0, 3.0)
        log_rho_bar = torch.log(rho_bar + 1e-10)
        s = (log_rho_bar * comp_mask).sum(dim=1, keepdim=True)
        n = comp_mask.sum(dim=1, keepdim=True).clamp(min=1)
        geo = torch.exp(s / n)
        rho_tilde = rho_bar / (geo + 1e-10)

        # Confidence gate on the same tokens
        c_gate = torch.sigmoid((ENTROPY_THRESHOLD - teach_ent) / GATE_TEMP)

        # Keep only positive-advantage tokens within completion mask
        A_row = adv_cpu.unsqueeze(1)              # (N, 1)
        valid = (comp_mask.bool()) & (A_row > 0)  # (N, T)

        if valid.sum() == 0:
            del gen, gen_in
            torch.cuda.empty_cache()
            continue

        rho_pos_list.append(rho_tilde[valid].numpy())
        c_pos_list.append(c_gate[valid].numpy())
        w_vals = 1.0 + c_gate[valid] * (rho_tilde[valid] - 1.0)
        w_pos_list.append(w_vals.numpy())
        adv_pos_list.append(A_row.expand_as(comp_mask)[valid].numpy())
        src_pos_list.append(np.array([ex["source"]] * int(valid.sum())))

        # capture count BEFORE del
        kept_pos = int(valid.sum())

        del gen, gen_in, ids, am, plen_t
        del old_logps, teach_logps, teach_ent, comp_mask
        del log_rho, rho, rho_bar, log_rho_bar, s, n, geo, rho_tilde, c_gate, valid
        torch.cuda.empty_cache()

        print(f"  [{pi+1}/{DIAG_N_PROBLEMS}] {ex['source']:6s} | "
              f"rewards={sum(rewards)}/{len(rewards)} | "
              f"kept {kept_pos} positive tokens")

# Concatenate
rho_pos = np.concatenate(rho_pos_list)
c_pos   = np.concatenate(c_pos_list)
w_pos   = np.concatenate(w_pos_list)
src_pos = np.concatenate(src_pos_list)

print(f"\nTotal positive-adv tokens: {len(rho_pos):,}")

# ========================= ANALYSIS =========================
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

def pct(a, q): return float(np.percentile(a, q))
def st(a):     return float(np.mean(a)), float(np.std(a))

# ---- rho_tilde distribution on positive tokens ----
print("\n--- rho_tilde on positive-advantage tokens ---")
print(f"  mean   = {rho_pos.mean():.4f}")
print(f"  std    = {rho_pos.std():.4f}")
print(f"  p10    = {pct(rho_pos, 10):.4f}")
print(f"  p50    = {pct(rho_pos, 50):.4f}")
print(f"  p90    = {pct(rho_pos, 90):.4f}")
print(f"  p99    = {pct(rho_pos, 99):.4f}")
print(f"  max    = {rho_pos.max():.4f}")
print(f"  frac rho > 1.2 = {(rho_pos > 1.2).mean():.4f}")
print(f"  frac rho < 0.8 = {(rho_pos < 0.8).mean():.4f}")
print(f"  mean |rho-1|   = {np.mean(np.abs(rho_pos - 1.0)):.4f}")

# ---- c distribution ----
print("\n--- confidence gate c on those same tokens ---")
print(f"  mean   = {c_pos.mean():.4f}")
print(f"  std    = {c_pos.std():.4f}")
print(f"  frac c > 0.9 = {(c_pos > 0.9).mean():.4f}")
print(f"  frac c < 0.1 = {(c_pos < 0.1).mean():.4f}")

# ---- effective w distribution ----
print("\n--- resulting weight w = 1 + c*(rho_tilde - 1) ---")
print(f"  mean   = {w_pos.mean():.4f}")
print(f"  std    = {w_pos.std():.4f}")
print(f"  mean |w-1| = {np.mean(np.abs(w_pos - 1.0)):.4f}")
print(f"  p90 |w-1|  = {pct(np.abs(w_pos - 1.0), 90):.4f}")

# ---- KEY: correlation between c and (rho_tilde - 1) ----
dev = rho_pos - 1.0
pearson = float(np.corrcoef(c_pos, dev)[0, 1])
print("\n--- ALIGNMENT CHECK ---")
print(f"  Pearson corr(c, rho_tilde - 1) = {pearson:+.4f}")
if pearson > 0.15:
    print("  >> POSITIVE: high-confidence tokens also have higher rho_tilde.")
    print("     Gate is aligned with the teacher signal. GOOD.")
elif pearson < -0.15:
    print("  >> NEGATIVE: high-confidence tokens have *lower* rho_tilde.")
    print("     Gate is suppressing the teacher's strong preferences.")
    print("     This is the WARNING case — check 2x2 table below.")
else:
    print("  >> NEAR ZERO: c and rho_tilde are roughly independent.")
    print("     Gate still useful (it removes tokens near rho=1 from being")
    print("     pushed further), but does not preferentially amplify.")

# ---- 2x2 contingency table ----
print("\n--- 2x2 alignment table on positive tokens ---")
# Quartile-like splits: c high if > 0.7, dev high if rho_tilde - 1 > 0.2
c_hi  = c_pos > 0.7
dev_hi = dev > 0.2

both_hi = (c_hi & dev_hi).mean()
c_hi_only = (c_hi & ~dev_hi).mean()
dev_hi_only = (~c_hi & dev_hi).mean()
neither = (~c_hi & ~dev_hi).mean()

print(f"  c>0.7 & rho>1.2 : {both_hi:.4f}   <- gate amplifies a strong teacher signal (GOOD)")
print(f"  c>0.7 & rho<=1.2: {c_hi_only:.4f}  <- gate trusts but teacher has little to say (neutral)")
print(f"  c<=0.7 & rho>1.2: {dev_hi_only:.4f} <- gate IGNORES a strong teacher signal (RISK)")
print(f"  c<=0.7 & rho<=1.2:{neither:.4f}   <- nothing happening (neutral)")

risk_ratio = dev_hi_only / max(both_hi, 1e-9)
print(f"\n  risk/benefit ratio = {risk_ratio:.3f}")
if risk_ratio > 0.5:
    print("  >> Ratio > 0.5 — the gate blocks a strong teacher signal about as")
    print("     often as it amplifies one. Consider softening (larger tau)")
    print("     or raising ENTROPY_THRESHOLD before training.")
else:
    print("  >> Ratio < 0.5 — gate mostly amplifies rather than suppresses.")
    print("     Safe to launch training with current calibration.")

# ---- per-source breakdown (if mixed pool) ----
if len(set(src_pos.tolist())) > 1:
    print("\n--- per-source breakdown ---")
    for src in sorted(set(src_pos.tolist())):
        m = src_pos == src
        print(f"  {src:10s} | n={m.sum():6d} | "
              f"rho_mean={rho_pos[m].mean():.3f} | "
              f"c_mean={c_pos[m].mean():.3f} | "
              f"|w-1|_mean={np.mean(np.abs(w_pos[m]-1)):.3f}")

# ========================= VERDICT =========================
print("\n" + "=" * 80)
print("VERDICT")
print("=" * 80)
if pearson > 0.1 and risk_ratio < 0.5:
    print("✅ Gate is aligned with the teacher signal — launch training.")
elif pearson < -0.1 or risk_ratio > 0.7:
    print("⚠️  Gate is misaligned — do NOT launch yet.")
    print("   Options:")
    print("     - raise ENTROPY_THRESHOLD (shift threshold up)")
    print("     - increase GATE_TEMP (soften transition)")
    print("     - consider gamma-power transform of rho_tilde")
else:
    print("➡  Gate is roughly orthogonal to rho_tilde.")
    print("   Training is safe, but expect limited *amplification* benefit;")
    print("   the gate will act mainly as a noise-suppressor.")
    print("   Fine to launch; report this in the thesis.")

# Save raw vectors for later plotting
np.savez("gate_alignment_diag.npz",
         rho=rho_pos, c=c_pos, w=w_pos, src=src_pos)
print("\nSaved raw vectors to gate_alignment_diag.npz (for plotting).")

✅ HF login OK
Loaded calibration: H* = 1.5737, tau = 0.3484, calib T = 1.2
Loading data...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

GPU0: 1.03 GB, GPU1: 5.70 GB

DIAGNOSTIC: rho_tilde vs confidence gate @ T=1.2
  [1/20] GSM8K  | rewards=7.0/10 | kept 1554 positive tokens
  [2/20] GSM8K  | rewards=3.0/10 | kept 795 positive tokens
  [4/20] GSM8K  | rewards=5.0/10 | kept 2560 positive tokens
  [6/20] GSM8K  | rewards=3.0/10 | kept 1086 positive tokens
  [7/20] GSM8K  | rewards=4.0/10 | kept 1124 positive tokens
  [8/20] GSM8K  | rewards=1.0/10 | kept 440 positive tokens
  [10/20] GSM8K  | rewards=1.0/10 | kept 512 positive tokens
  [11/20] GSM8K  | rewards=5.0/10 | kept 1425 positive tokens
  [12/20] GSM8K  | rewards=2.0/10 | kept 774 positive tokens
  [15/20] GSM8K  | rewards=6.0/10 | kept 756 positive tokens
  [17/20] GSM8K  | rewards=1.0/10 | kept 375 positive tokens
  [19/20] GSM8K  | rewards=6.0/10 | kept 3072 positive tokens

Total positive-adv tokens: 14,473

RESULTS

--- rho_tilde on positive-advantage tokens ---
  mean   = 1.2270
  std    = 1.0326
  p10    = 0.6066
  p50    = 0.7732
  p90    = 2.2330
  p99  